In [ ]:
# compare_models_updated.py
"""
Updated compare_models:
 - Train: ResNet50, EfficientNet-B4, ViT (pretrained)
 - Do NOT train Hybrid (load checkpoint and evaluate)
 - Use masked dataset when masks exist (image_mask.png present) to ignore corners
 - Save metrics, plots, and checkpoints
"""

import os
import copy
import time
import csv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image
import timm
import cv2

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score
)
from sklearn.preprocessing import label_binarize

from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from tqdm import tqdm

# -----------------------------
# CONFIG - EDIT THESE
# -----------------------------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Standard ImageFolder layout with train/val/test (if you use masked cleaned folder, put masks next to images)
CLEAN_ROOT = r"D:\own_cleaned_mask"    # optional masked cleaned dataset (train/val/test/class/*.jpg + *_mask.png)
USE_MASKED_IF_AVAILABLE = True         # if True and masks exist, use MaskedImageFolder

OUTPUT_DIR = "checkpoints_compare_v2"
PLOT_DIR = "plots_compare_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

HYBRID_CHECKPOINT = "best_reseff_fusion_masked.pth"  # your existing hybrid checkpoint (won't be trained)

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 4
NUM_EPOCHS = 100
LR = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 10  # early stopping on val loss

# -----------------------------
# Mask-aware Dataset (optional)
# -----------------------------
class MaskedImageFolder(Dataset):
    """
    Expects path structure:
      root/train/class_x/*.jpg
      root/train/class_x/*_mask.png
    If mask missing, falls back to all-ones mask.
    Returns normalized tensors.
    """
    def __init__(self, root, img_size=IMG_SIZE):
        self.root = root
        classes = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
        if len(classes) == 0:
            raise RuntimeError(f"No class folders found in {root}")
        self.classes = classes
        self.class_to_idx = {c:i for i,c in enumerate(classes)}
        self.samples = []
        for c in classes:
            cls_dir = os.path.join(root, c)
            for fname in sorted(os.listdir(cls_dir)):
                if not fname.lower().endswith(('.jpg','.jpeg','.png')):
                    continue
                if fname.endswith("_mask.png"):
                    continue
                img_path = os.path.join(cls_dir, fname)
                base = os.path.splitext(fname)[0]
                mask_path = os.path.join(cls_dir, f"{base}_mask.png")
                if not os.path.exists(mask_path):
                    mask_path = None
                self.samples.append((img_path, mask_path, self.class_to_idx[c]))

        self.img_size = img_size
        self.to_tensor = transforms.ToTensor()
        self.resize = transforms.Resize((img_size, img_size))
        self.mean_tensor = torch.tensor(IMAGENET_DEFAULT_MEAN).view(3,1,1).float()

    def __len__(self):
        return len(self.samples)

    def _load_mask(self, mask_path):
        if mask_path is None:
            return np.ones((self.img_size, self.img_size), dtype=np.uint8)
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            return np.ones((self.img_size, self.img_size), dtype=np.uint8)
        m = cv2.resize(m, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        return (m > 127).astype(np.uint8)

    def __getitem__(self, idx):
        img_path, mask_path, label = self.samples[idx]
        pil = Image.open(img_path).convert("RGB")
        pil = self.resize(pil)
        img_tensor = self.to_tensor(pil)  # [0..1]
        mask_np = self._load_mask(mask_path)
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0).float()
        # replace masked-out pixels with ImageNet mean (0..1)
        img_tensor = img_tensor * mask_tensor + (1.0 - mask_tensor) * self.mean_tensor
        img_tensor = transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)(img_tensor)
        return img_tensor, label

# -----------------------------
# Helper: choose dataset loader (masked if available)
# -----------------------------
def build_dataloaders(data_root, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    """
    data_root: root with train/val/test subfolders
    If masks exist in data_root, uses MaskedImageFolder; otherwise ImageFolder.
    """
    train_dir = os.path.join(data_root, "train")
    val_dir = os.path.join(data_root, "val")
    test_dir = os.path.join(data_root, "test")

    # quick check for masks in train dir
    use_masked = False
    try:
        for cls in os.listdir(train_dir):
            cls_path = os.path.join(train_dir, cls)
            if os.path.isdir(cls_path):
                # look for any _mask.png file
                for f in os.listdir(cls_path):
                    if f.endswith("_mask.png"):
                        use_masked = True
                        break
            if use_masked:
                break
    except Exception:
        use_masked = False

    if USE_MASKED_IF_AVAILABLE and use_masked:
        print("Using MaskedImageFolder (masks detected).")
        train_ds = MaskedImageFolder(train_dir)
        val_ds = MaskedImageFolder(val_dir)
        test_ds = MaskedImageFolder(test_dir)
    else:
        print("Using standard ImageFolder (no masks detected).")
        train_tf = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(0.1,0.1,0.1,0.05),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
        ])
        eval_tf = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
        ])
        train_ds = ImageFolder(train_dir, transform=train_tf)
        val_ds = ImageFolder(val_dir, transform=eval_tf)
        test_ds = ImageFolder(test_dir, transform=eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size*2, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size*2, shuffle=False, num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds

# -----------------------------
# Model factories (pretrained)
# -----------------------------
class ResNetOnly(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = timm.create_model('resnet50', pretrained=True, num_classes=num_classes)
    def forward(self, x): return self.model(x)

class EfficientNetOnly(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=num_classes)
    def forward(self, x): return self.model(x)

class ViTOnly(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes)
    def forward(self, x): return self.model(x)

# Hybrid definition only for loading / evaluation (not training)
class ResEffFusion(nn.Module):
    def __init__(self, num_classes, eff_weight=0.75):
        super().__init__()
        self.eff_weight = eff_weight
        self.res_weight = 1.0 - eff_weight
        self.eff = timm.create_model("efficientnet_b4", pretrained=False, features_only=True)
        eff_dim = self.eff.feature_info[-1]['num_chs']
        self.res = timm.create_model("resnet50", pretrained=False, features_only=True)
        res_dim = self.res.feature_info[-1]['num_chs']
        self.eff_proj = nn.Conv2d(eff_dim, 1024, kernel_size=1)
        self.res_proj = nn.Conv2d(res_dim, 1024, kernel_size=1)
        self.bn = nn.BatchNorm2d(1024)
        self.relu = nn.ReLU(inplace=False)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(1024, num_classes)
    def forward(self, x):
        eff_feat = self.eff(x)[-1]
        res_feat = self.res(x)[-1]
        if eff_feat.shape[2:] != res_feat.shape[2:]:
            res_feat = nn.functional.interpolate(res_feat, size=eff_feat.shape[2:], mode="bilinear", align_corners=False)
        eff_feat = self.eff_proj(eff_feat)
        res_feat = self.res_proj(res_feat)
        fused = self.eff_weight * eff_feat + self.res_weight * res_feat
        fused = self.relu(self.bn(fused))
        pooled = self.pool(fused).flatten(1)
        return self.classifier(pooled)

# -----------------------------
# Training utilities (train one model)
# -----------------------------
def train_one_model(model, name, train_loader, val_loader, num_epochs=NUM_EPOCHS, output_dir=OUTPUT_DIR):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))

    best_w = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    bad_epochs = 0

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, num_epochs+1):
        model.train()
        running_loss = 0.0
        running_corrects = 0
        n = 0
        pbar = tqdm(train_loader, desc=f"[{name}] Epoch {epoch}/{num_epochs}")
        for imgs, labels in pbar:
            imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            preds = torch.argmax(outputs, dim=1)
            running_loss += loss.item() * imgs.size(0)
            running_corrects += (preds == labels).sum().item()
            n += imgs.size(0)
            pbar.set_postfix({'loss': f"{running_loss/max(1,n):.4f}"})

        epoch_train_loss = running_loss / max(1, n)
        epoch_train_acc = running_corrects / max(1, n)

        # validation
        model.eval()
        v_loss = 0.0
        v_corrects = 0
        v_n = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                preds = torch.argmax(outputs, dim=1)
                v_loss += loss.item() * imgs.size(0)
                v_corrects += (preds == labels).sum().item()
                v_n += imgs.size(0)

        epoch_val_loss = v_loss / max(1, v_n)
        epoch_val_acc = v_corrects / max(1, v_n)

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

        scheduler.step()

        print(f"[{name}] Epoch {epoch} -> train_loss={epoch_train_loss:.4f} train_acc={epoch_train_acc:.4f} val_loss={epoch_val_loss:.4f} val_acc={epoch_val_acc:.4f}")

        # early stopping by val_loss
        if epoch_val_loss + 1e-8 < best_val_loss:
            best_val_loss = epoch_val_loss
            best_w = copy.deepcopy(model.state_dict())
            bad_epochs = 0
            torch.save(best_w, os.path.join(output_dir, f"{name}_best.pth"))
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"[{name}] Early stopping at epoch {epoch} (no val_loss improvement for {PATIENCE} epochs).")
                break

    # restore best
    model.load_state_dict(best_w)
    torch.save(model.state_dict(), os.path.join(output_dir, f"{name}_final.pth"))
    return model, history

# -----------------------------
# Evaluation utility (full metrics and plots)
# -----------------------------
def evaluate_and_plot(model, loader, class_names, model_name, plot_dir=PLOT_DIR):
    model.eval()
    y_true = []
    y_pred = []
    y_scores = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds)
            y_scores.extend(probs)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_scores = np.array(y_scores)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    macro_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)

    report_text = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)

    # confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{model_name} - Confusion Matrix")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f"{model_name}_cm.png"))
    plt.close()

    # normalized cm
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-12)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap='Greens', xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{model_name} - Normalized CM")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f"{model_name}_cm_norm.png"))
    plt.close()

    # ROC and PR if possible
    try:
        y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
        plt.figure(figsize=(8,6))
        for i, cls in enumerate(class_names):
            try:
                from sklearn.metrics import roc_curve, auc, precision_recall_curve
                fpr, tpr, _ = roc_curve(y_bin[:,i], y_scores[:,i])
                auc_val = roc_auc_score(y_bin[:,i], y_scores[:,i])
                plt.plot(fpr, tpr, label=f"{cls} (AUC={auc_val:.3f})")
            except Exception:
                continue
        plt.plot([0,1],[0,1],'k--', alpha=0.3)
        plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"{model_name} - ROC Curves")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, f"{model_name}_roc.png"))
        plt.close()

        plt.figure(figsize=(8,6))
        for i, cls in enumerate(class_names):
            try:
                precision, recall, _ = precision_recall_curve(y_bin[:,i], y_scores[:,i])
                ap = average_precision_score(y_bin[:,i], y_scores[:,i])
                plt.plot(recall, precision, label=f"{cls} (AP={ap:.3f})")
            except Exception:
                continue
        plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"{model_name} - PR Curves")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, f"{model_name}_pr.png"))
        plt.close()
    except Exception as e:
        print("Warning: ROC/PR plotting failed:", e)

    stats = {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "classification_report": report_text,
        "confusion_matrix": cm
    }
    return stats, y_true, y_pred, y_scores

# -----------------------------
# MAIN
# -----------------------------
def main():
    # Build dataloaders (prefer CLEAN_ROOT if masks present)
    # prefer CLEAN_ROOT only if masks detected in its train subfolders
    data_root_to_use = CLEAN_ROOT if (USE_MASKED_IF_AVAILABLE and os.path.exists(CLEAN_ROOT)) else DATA_ROOT
    train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = build_dataloaders(data_root_to_use, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

    CLASS_NAMES = train_ds.classes if hasattr(train_ds, "classes") else train_ds.classes
    NUM_CLASSES = len(CLASS_NAMES)
    print("Detected classes:", CLASS_NAMES)

    # Models to train (no hybrid)
    models = {
        "ResNet50": ResNetOnly(NUM_CLASSES),
        "EfficientNet-B4": EfficientNetOnly(NUM_CLASSES),
        "ViT": ViTOnly(NUM_CLASSES)
    }

    summary_rows = []

    for name, model in models.items():
        print("\n" + "="*60)
        print(f"TRAINING {name}")
        print("="*60)
        trained_model, history = train_one_model(model, name, train_loader, val_loader, num_epochs=NUM_EPOCHS, output_dir=OUTPUT_DIR)
        print(f"Evaluating {name} on test set")
        stats, y_true, y_pred, y_scores = evaluate_and_plot(trained_model, test_loader, CLASS_NAMES, name, plot_dir=PLOT_DIR)
        params = sum(p.numel() for p in trained_model.parameters() if p.requires_grad)
        summary_rows.append({
            "model": name,
            "accuracy": stats["accuracy"],
            "macro_f1": stats["macro_f1"],
            "macro_precision": stats["macro_precision"],
            "macro_recall": stats["macro_recall"],
            "params": int(params)
        })

    # Now evaluate your pretrained Hybrid model (do NOT train it)
    if os.path.exists(HYBRID_CHECKPOINT):
        print("\n" + "="*60)
        print("LOADING & EVALUATING HYBRID (no training)")
        print("="*60)
        hybrid = ResEffFusion(NUM_CLASSES)
        # load hybrid checkpoint (state_dict must match)
        hybrid.load_state_dict(torch.load(HYBRID_CHECKPOINT, map_location=DEVICE))
        hybrid = hybrid.to(DEVICE)
        stats_h, y_true_h, y_pred_h, y_scores_h = evaluate_and_plot(hybrid, test_loader, CLASS_NAMES, "Hybrid", plot_dir=PLOT_DIR)
        params_h = sum(p.numel() for p in hybrid.parameters() if p.requires_grad)
        summary_rows.append({
            "model": "Hybrid",
            "accuracy": stats_h["accuracy"],
            "macro_f1": stats_h["macro_f1"],
            "macro_precision": stats_h["macro_precision"],
            "macro_recall": stats_h["macro_recall"],
            "params": int(params_h)
        })
    else:
        print("Hybrid checkpoint not found at:", HYBRID_CHECKPOINT)
        print("Skipping Hybrid evaluation.")

    # Save summary CSV and print
    csv_path = os.path.join(OUTPUT_DIR, "models_comparison_summary.csv")
    keys = summary_rows[0].keys() if summary_rows else ["model","accuracy"]
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(keys))
        writer.writeheader()
        writer.writerows(summary_rows)

    import pandas as pd
    df = pd.DataFrame(summary_rows).sort_values(by="macro_f1", ascending=False)
    print("\nFINAL COMPARISON")
    print(df.to_string(index=False))
    print(f"\nSaved summary CSV to {csv_path}. Plots saved to {PLOT_DIR}, checkpoints to {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

Using MaskedImageFolder (masks detected).
Detected classes: ['0_normal', '1_ulcerative_colitis', '2_polyps', '3_esophagitis']


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4224\1841609963.py:239: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))



TRAINING ResNet50


[ResNet50] Epoch 1/100:   0%|          | 0/323 [00:00<?, ?it/s]

In [2]:
# ============================================================
# IMPORTS
# ============================================================
import os
import cv2
import torch
import timm
import numpy as np
import torch.nn as nn
from PIL import Image

from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score
)
from sklearn.preprocessing import label_binarize
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD

# ============================================================
# CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_SIZE = 224
BATCH_SIZE = 16

DATA_ROOT = r"D:\own_cleaned_mask\test"  # masked test set
CHECKPOINT_DIR = "checkpoints_compare_v2"  # where your checkpoints are saved (for loading and evaluation)

# ============================================================
# MASKED DATASET
# ============================================================

class MaskedImageFolder(Dataset):

    def __init__(self, root):
        self.samples = []
        self.classes = sorted([
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        ])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        for cls in self.classes:
            cls_dir = os.path.join(root, cls)
            for fname in os.listdir(cls_dir):

                if fname.endswith("_mask.png"):
                    continue
                if not fname.lower().endswith(('.jpg','.jpeg','.png')):
                    continue

                img_path = os.path.join(cls_dir, fname)
                base = os.path.splitext(fname)[0]
                mask_path = os.path.join(cls_dir, f"{base}_mask.png")

                if not os.path.exists(mask_path):
                    mask_path = None

                self.samples.append((img_path, mask_path, self.class_to_idx[cls]))

        self.resize = transforms.Resize((IMG_SIZE, IMG_SIZE))
        self.to_tensor = transforms.ToTensor()
        self.mean_tensor = torch.tensor(IMAGENET_DEFAULT_MEAN).view(3,1,1).float()

    def __len__(self):
        return len(self.samples)

    def _load_mask(self, path):
        if path is None:
            return np.ones((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

        m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            return np.ones((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

        m = cv2.resize(m, (IMG_SIZE, IMG_SIZE))
        return (m > 127).astype(np.uint8)

    def __getitem__(self, idx):
        img_path, mask_path, label = self.samples[idx]

        pil = Image.open(img_path).convert("RGB")
        pil = self.resize(pil)

        img_tensor = self.to_tensor(pil)

        mask = self._load_mask(mask_path)
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()

        img_tensor = img_tensor * mask_tensor + (1-mask_tensor) * self.mean_tensor
        img_tensor = transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)(img_tensor)

        return img_tensor, label


# ============================================================
# MODELS
# ============================================================

class ResNetOnly(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = timm.create_model("resnet50", pretrained=False, num_classes=num_classes)
    def forward(self, x):
        return self.model(x)


class EfficientNetOnly(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = timm.create_model("efficientnet_b4", pretrained=False, num_classes=num_classes)
    def forward(self, x):
        return self.model(x)


class ViTOnly(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=num_classes)
    def forward(self, x):
        return self.model(x)


class ResEffFusion(nn.Module):
    def __init__(self, num_classes, eff_weight=0.75):
        super().__init__()

        self.eff_weight = eff_weight
        self.res_weight = 1 - eff_weight

        self.eff = timm.create_model("efficientnet_b4", pretrained=False, features_only=True)
        eff_dim = self.eff.feature_info[-1]['num_chs']

        self.res = timm.create_model("resnet50", pretrained=False, features_only=True)
        res_dim = self.res.feature_info[-1]['num_chs']

        self.eff_proj = nn.Conv2d(eff_dim, 1024, 1)
        self.res_proj = nn.Conv2d(res_dim, 1024, 1)

        self.bn = nn.BatchNorm2d(1024)
        self.relu = nn.ReLU(inplace=False)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(1024, num_classes)

    def forward(self, x):
        eff = self.eff(x)[-1]
        res = self.res(x)[-1]

        if eff.shape[2:] != res.shape[2:]:
            res = nn.functional.interpolate(res, size=eff.shape[2:], mode="bilinear", align_corners=False)

        eff = self.eff_proj(eff)
        res = self.res_proj(res)

        fused = self.eff_weight * eff + self.res_weight * res
        fused = self.relu(self.bn(fused))

        pooled = self.pool(fused).flatten(1)
        return self.classifier(pooled)


# ============================================================
# LOAD DATA
# ============================================================

test_dataset = MaskedImageFolder(DATA_ROOT)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

CLASS_NAMES = test_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print("Classes:", CLASS_NAMES)

# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_model(model, checkpoint_name):

    model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, checkpoint_name), map_location=DEVICE))
    model.to(DEVICE)
    model.eval()

    y_true = []
    y_pred = []
    y_probs = []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)

            preds = torch.argmax(probs, dim=1)

            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
            y_probs.extend(probs.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_probs = np.array(y_probs)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)

    # ROC & PR AUC (One-vs-Rest)
    try:
        y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
        roc_auc = roc_auc_score(y_bin, y_probs, average="macro", multi_class="ovr")
        pr_auc = average_precision_score(y_bin, y_probs, average="macro")
    except:
        roc_auc = float("nan")
        pr_auc = float("nan")

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_roc_auc": roc_auc,
        "macro_pr_auc": pr_auc
    }


# ============================================================
# RUN ALL MODELS
# ============================================================

models = {
    "ResNet50": (ResNetOnly(NUM_CLASSES), "ResNet50_best.pth"),
    "EfficientNet-B4": (EfficientNetOnly(NUM_CLASSES), "EfficientNet-B4_best.pth"),
    "ViT": (ViTOnly(NUM_CLASSES), "ViT_best.pth"),
    "EffResFusion": (ResEffFusion(NUM_CLASSES), "EffResFusion_best.pth")
}

results = {}

for name, (model, ckpt) in models.items():
    print(f"\nEvaluating {name}...")
    results[name] = evaluate_model(model, ckpt)


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n" + "="*70)
print("FINAL PERFORMANCE COMPARISON")
print("="*70)

for name, metrics in results.items():
    print(f"\n{name}")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

Classes: ['0_normal', '1_ulcerative_colitis', '2_polyps', '3_esophagitis']

Evaluating ResNet50...

Evaluating EfficientNet-B4...

Evaluating ViT...

Evaluating EffResFusion...

FINAL PERFORMANCE COMPARISON

ResNet50
accuracy: 0.9802
macro_f1: 0.9816
macro_precision: 0.9806
macro_recall: 0.9827
macro_roc_auc: 0.9994
macro_pr_auc: 0.9985

EfficientNet-B4
accuracy: 0.9829
macro_f1: 0.9848
macro_precision: 0.9851
macro_recall: 0.9846
macro_roc_auc: 0.9995
macro_pr_auc: 0.9988

ViT
accuracy: 0.8855
macro_f1: 0.8974
macro_precision: 0.8953
macro_recall: 0.9023
macro_roc_auc: 0.9798
macro_pr_auc: 0.9563

EffResFusion
accuracy: 0.9847
macro_f1: 0.9854
macro_precision: 0.9856
macro_recall: 0.9853
macro_roc_auc: 0.9997
macro_pr_auc: 0.9993
